# Создание и заполнение данных БД Postgre

In [2]:
%pip install python-dotenv psycopg2-binary
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import json
import psycopg2
import pandas as pd
from psycopg2.extras import DictCursor
from dotenv import load_dotenv


# Получение секретов

In [5]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=Davydoff, DB=my_db_Davydoff, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [7]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost", # если Docker контейнер запущен локально, а ноутбук вне Docker.
                          # НО! если ноутбук также в Docker и в одной сети с БД,
                          # то нужно использовать имя сервиса Docker (например, 'db' или 'postgres_db').
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [8]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 13.23 (Debian 13.23-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)


In [9]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- departments
- user_logs


In [10]:
# закрытие соединения с БД - После завершения работы с БД не забываем закрывать соединение!
cursor.close()
conn.close()

Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура, специалитет, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки. 

In [31]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=Davydoff, DB=my_db_Davydoff, DB_PORT=5433


In [ ]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost",
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

conn.autocommit = True

Успешное подключение к базе данных!


In [15]:
query = "SELECT * FROM USER_LOGS ORDER BY RANDOM() LIMIT 10"
cursor.execute(query)
rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]
print(f"Колонки: {columns}\n")

for row in rows:
    print(row)

Колонки: ['courseid', 'userid', 'num_week', 's_all', 's_all_avg', 's_course_viewed', 's_course_viewed_avg', 's_q_attempt_viewed', 's_q_attempt_viewed_avg', 's_a_course_module_viewed', 's_a_course_module_viewed_avg', 's_a_submission_status_viewed', 's_a_submission_status_viewed_avg', 'namer_level', 'name_vatt', 'depart', 'name_osno', 'name_formopril', 'leveled', 'num_sem', 'kurs', 'date_vatt']

(71987, 32351, 25, 4, 9.25, 2, 4.6, 0, 0.0, 1, 2.3, 1, 2.3, 4, 'Экзамен', 21, 2, 2, 1, 4, 3, datetime.date(2022, 6, 29))
(76293, 33729, 22, 1, 1.0588, 1, 0.3529, 0, 0.0, 0, 0.0588, 0, 0.0588, 2, 'Экзамен', 41, 2, 1, 1, 2, 2, datetime.date(2022, 6, 27))
(72359, 30743, 22, 0, 0.1176, 0, 0.0588, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 16, 1, 1, 1, 4, 3, datetime.date(2022, 6, 20))
(73090, 21392, 8, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 17, 2, 2, 1, 6, 4, datetime.date(2022, 6, 15))
(78272, 23605, 23, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 5, 'Экзамен', 11, 2, 2, 1, 8, 5, datetime.date

In [16]:
query = "SELECT * FROM USER_LOGS ORDER BY RANDOM() LIMIT 10"
df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_16800\1171977693.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,courseid,userid,num_week,s_all,s_all_avg,s_course_viewed,s_course_viewed_avg,s_q_attempt_viewed,s_q_attempt_viewed_avg,s_a_course_module_viewed,...,s_a_submission_status_viewed_avg,namer_level,name_vatt,depart,name_osno,name_formopril,leveled,num_sem,kurs,date_vatt
0,88721,36728,10,0,0.0000,0,0.0000,0,0.0,0,...,0.0000,2,Экзамен,43,2,2,1,2,2,2022-06-29
1,88347,24500,19,0,0.0000,0,0.0000,0,0.0,0,...,0.0000,2,Экзамен,30,2,1,1,6,4,2022-06-22
2,74969,33668,22,0,0.1176,0,0.1176,0,0.0,0,...,0.0000,5,Экзамен,3,1,1,3,2,2,2022-06-25
3,76568,25994,16,55,15.0909,18,4.4545,0,0.0,14,...,3.6364,3,Экзамен,24,2,1,2,8,5,2022-06-27
4,72232,27685,21,0,0.6250,0,0.6250,0,0.0,0,...,0.0000,3,Экзамен,37,1,2,1,6,4,2022-06-24
5,88856,35662,16,2,5.0000,2,2.7273,0,0.0,0,...,0.5455,4,Экзамен,22,1,1,1,2,2,2022-06-27
6,75833,36046,6,1,1.0000,1,1.0000,0,0.0,0,...,0.0000,2,Экзамен,8,1,1,2,2,2,2022-06-18
7,84979,23135,11,0,0.0000,0,0.0000,0,0.0,0,...,0.0000,4,Экзамен,9,2,2,1,8,5,2022-06-20
8,71571,33391,29,0,5.7917,0,0.8750,0,3.0,0,...,0.0833,4,Экзамен,20,1,1,1,2,2,2022-06-21
9,88979,34149,16,0,1.0000,0,0.6364,0,0.0,0,...,0.0000,5,Экзамен,4,1,1,3,2,2,2022-07-06


## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [17]:
query = """
    SELECT 
        COUNT(DISTINCT depart) AS departments_count
    FROM USER_LOGS
    WHERE courseid IS NOT NULL
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_16800\3608507271.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,departments_count
0,43


##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводить сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [20]:
query = """
    SELECT 
        DEPARTMENTS.name AS DEPTNAME,
        COUNT(USER_LOGS.courseid) AS COUNTCOURSE
    FROM USER_LOGS INNER JOIN DEPARTMENTS ON USER_LOGS.depart = DEPARTMENTS.id
    GROUP BY DEPARTMENTS.name
    ORDER BY COUNTCOURSE DESC
"""

df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_16800\1262581724.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,deptname,countcourse
0,МиХТ,25296
1,ДиСО,25176
2,ЛиП,19008
3,РМПИ,17856
4,ГМДиОПИ,16968
5,ТОМ,16704
6,БИиИТ,16176
7,ПОиД,14760
8,АЭПиМ,14232
9,ЛиУТС,13440


## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [32]:
query = """

    SELECT 
        COUNT(DEPTCOUNT.courseid)
    FROM 
    (
        SELECT
            USER_LOGS.courseid,
            COUNT(DISTINCT USER_LOGS.depart) AS departments_count
        FROM USER_LOGS
        WHERE courseid IS NOT NULL
        GROUP BY USER_LOGS.courseid
        HAVING COUNT(DISTINCT depart) > 1
    ) AS DEPTCOUNT 
"""



df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_16800\866560846.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,count
0,60


In [ ]:
#получили курсы, на которые закреплены несколько кафедр
query = """
    SELECT
        USER_LOGS.courseid AS COURSEID,
        COUNT(DISTINCT USER_LOGS.depart) AS DEPCOUNT
    FROM
        USER_LOGS
    GROUP BY USER_LOGS.courseid
    HAVING COUNT(DISTINCT USER_LOGS.depart) > 1
"""

#мы должны теперь найти уникальные кафедры, сравнивая их курсы с курсами, имеющими несколько кафедр
#тем самым на каждый курс получим кафедры, которые закреплены именно на один курс.
query = """

    WITH COURSRES AS 
    (
	SELECT
		USER_LOGS.courseid AS COURSEID,
		COUNT(DISTINCT USER_LOGS.depart) AS DEPCOUNT
	FROM
		USER_LOGS
	GROUP BY USER_LOGS.courseid
	HAVING COUNT(DISTINCT USER_LOGS.depart) > 1
    )

    SELECT DISTINCT
	    DEPARTMENTS.name,
	    UL.courseid
    FROM USER_LOGS AS UL
	    INNER JOIN DEPARTMENTS ON UL.DEPART = DEPARTMENTS.ID
	    INNER JOIN COURSRES ON UL.COURSEID = COURSRES.COURSEID
"""


df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_16800\874770190.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,courseid,depcount
0,71495,3
1,71508,2
2,71541,2
3,71547,2
4,71549,2
5,71571,3
6,71632,2
7,71736,2
8,71852,2
9,71857,2


## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за вест период обучения).

## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников. 

## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |